# Aula 08: MongoDB Aggregation Framework
### Tecnologia em Análise e Desenvolvimento de Sistemas (ADS) — IFCE Campus Tauá
**Componente:** Banco de Dados Não-Relacionais (ADS16)  
**Docente:** Prof. Me. Reginaldo Fernandes

---

## 🎯 Objetivos Deste Caderno Interativo
1. Compreender a arquitetura de **Pipeline de Agregação (Dataflow)** do MongoDB.
2. Comparar operadores de agregação do MongoDB com cláusulas SQL (`WHERE`, `GROUP BY`, `HAVING`, `JOIN`).
3. Executar pipelines analíticos reais sobre as coleções de e-commerce: `pedidos`, `produtos`, `clientes` e `avaliacoes`.
4. Medir o impacto de otimizações de memória RAM e filtros precoces com `$match` usando índices $B\text{-Tree}$.

In [ ]:
from pymongo import MongoClient
import pandas as pd

# Conectando à instância local do MongoDB
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=2000)
db = client["ecommerce_shopgrids"]
print("Coleções disponíveis no ShopGrids:", db.list_collection_names())
print(f"• Clientes:   {db.clientes.count_documents({}):,}")
print(f"• Produtos:   {db.produtos.count_documents({}):,}")
print(f"• Pedidos:    {db.pedidos.count_documents({}):,}")
print(f"• Avaliações: {db.avaliacoes.count_documents({}):,}")

## 1. Faturamento Mensal e Ticket Médio
Uso encadeado dos estágios `$match` $\to$ `$group` $\to$ `$project` $\to$ `$sort`.

In [ ]:
pipeline_faturamento = [
    {"$match": {"status": "ENTREGUE"}},
    {
        "$group": {
            "_id": {"ano": {"$year": "$data_criacao"}, "mes": {"$month": "$data_criacao"}},
            "faturamento": {"$sum": "$pagamento.total_pago"},
            "ticket_medio": {"$avg": "$pagamento.total_pago"},
            "total_pedidos": {"$sum": 1}
        }
    },
    {
        "$project": {
            "_id": 0,
            "periodo": {
                "$concat": [
                    {"$toString": "$_id.ano"}, "-",
                    {"$cond": [{"$lt": ["$_id.mes", 10]}, {"$concat": ["0", {"$toString": "$_id.mes"}]}, {"$toString": "$_id.mes"}]}
                ]
            },
            "faturamento": {"$round": ["$faturamento", 2]},
            "ticket_medio": {"$round": ["$ticket_medio", 2]},
            "total_pedidos": 1
        }
    },
    {"$sort": {"periodo": 1}}
]

df_faturamento = pd.DataFrame(list(db.pedidos.aggregate(pipeline_faturamento)))
df_faturamento

## 2. Desconstrução de Arrays com `$unwind`: Produtos Mais Vendidos
Cada documento de pedido contém uma lista de itens. O `$unwind` expande a lista em fluxos individuais.

In [ ]:
pipeline_produtos = [
    {"$match": {"status": "ENTREGUE"}},
    {"$unwind": "$itens"},
    {
        "$group": {
            "_id": "$itens.sku",
            "quantidade": {"$sum": "$itens.qtd"},
            "receita": {"$sum": {"$multiply": ["$itens.preco", "$itens.qtd"]}}
        }
    },
    {
        "$project": {
            "_id": 0,
            "sku": "$_id",
            "quantidade": 1,
            "receita": {"$round": ["$receita", 2]}
        }
    },
    {"$sort": {"receita": -1}},
    {"$limit": 5}
]

pd.DataFrame(list(db.pedidos.aggregate(pipeline_produtos)))

## 3. Junção Externa com `$lookup` entre Pedidos e Catálogo de Produtos
Cruzando os itens desconstruídos de `pedidos` com a coleção de referência `produtos` para obter título e categoria.

In [ ]:
pipeline_lookup_produtos = [
    {"$match": {"status": "ENTREGUE"}},
    {"$unwind": "$itens"},
    {
        "$lookup": {
            "from": "produtos",
            "localField": "itens.sku",
            "foreignField": "sku",
            "as": "detalhes"
        }
    },
    {"$unwind": "$detalhes"},
    {
        "$group": {
            "_id": {
                "sku": "$itens.sku",
                "titulo": "$detalhes.titulo",
                "categoria": "$detalhes.categoria"
            },
            "unidades": {"$sum": "$itens.qtd"},
            "receita_total": {"$sum": {"$multiply": ["$itens.preco", "$itens.qtd"]}}
        }
    },
    {
        "$project": {
            "_id": 0,
            "sku": "$_id.sku",
            "titulo": "$_id.titulo",
            "categoria": "$_id.categoria",
            "unidades": 1,
            "receita_total": {"$round": ["$receita_total", 2]}
        }
    },
    {"$sort": {"receita_total": -1}},
    {"$limit": 5}
]

pd.DataFrame(list(db.pedidos.aggregate(pipeline_lookup_produtos)))

## 4. Junção Externa com `$lookup` entre Pedidos e Cadastro de Clientes
Enriquecendo dados transacionais com informações cadastrais (`nome`, `cidade`, `estado`, `segmento`).

In [ ]:
pipeline_lookup_clientes = [
    {"$match": {"status": "ENTREGUE"}},
    {"$sort": {"pagamento.total_pago": -1}},
    {"$limit": 5},
    {
        "$lookup": {
            "from": "clientes",
            "localField": "cliente_id",
            "foreignField": "cliente_id",
            "as": "cliente"
        }
    },
    {"$unwind": "$cliente"},
    {
        "$project": {
            "_id": 0,
            "codigo": 1,
            "cliente": "$cliente.nome",
            "cidade_uf": {"$concat": ["$cliente.cidade", "/", "$cliente.estado"]},
            "segmento": "$cliente.segmento",
            "total_pago": "$pagamento.total_pago"
        }
    }
]

pd.DataFrame(list(db.pedidos.aggregate(pipeline_lookup_clientes)))

## 5. Junção Externa com `$lookup` entre Catálogo e Avaliações de Clientes
Cruzando `produtos` com a coleção `avaliacoes` para calcular nota média e quantidade de feedbacks por produto.

In [ ]:
pipeline_lookup_avaliacoes = [
    {"$match": {"categoria": "Informática", "status": "ATIVO"}},
    {
        "$lookup": {
            "from": "avaliacoes",
            "localField": "sku",
            "foreignField": "sku",
            "as": "reviews"
        }
    },
    {
        "$project": {
            "_id": 0,
            "sku": 1,
            "titulo": 1,
            "preco": 1,
            "total_avaliacoes": {"$size": "$reviews"},
            "nota_media": {"$round": [{"$avg": "$reviews.nota"}, 1]}
        }
    },
    {"$match": {"total_avaliacoes": {"$gte": 2}}},
    {"$sort": {"nota_media": -1, "total_avaliacoes": -1}},
    {"$limit": 5}
]

pd.DataFrame(list(db.produtos.aggregate(pipeline_lookup_avaliacoes)))

## 6. Painel Executivo Consolidado com `$facet`
Múltiplos fluxos analíticos em paralelo em uma única consulta de ida e volta ao servidor (*Single Round-Trip*).

In [ ]:
pipeline_facet = [
    {"$match": {"status": "ENTREGUE"}},
    {
        "$facet": {
            "kpis": [
                {
                    "$group": {
                        "_id": None,
                        "faturamento_global": {"$sum": "$pagamento.total_pago"},
                        "total_pedidos": {"$sum": 1},
                        "ticket_medio": {"$avg": "$pagamento.total_pago"}
                    }
                },
                {
                    "$project": {
                        "_id": 0,
                        "faturamento_global": {"$round": ["$faturamento_global", 2]},
                        "total_pedidos": 1,
                        "ticket_medio": {"$round": ["$ticket_medio", 2]}
                    }
                }
            ],
            "maiores_pedidos": [
                {"$sort": {"pagamento.total_pago": -1}},
                {"$limit": 3},
                {
                    "$project": {
                        "_id": 0,
                        "codigo": 1,
                        "cliente_id": 1,
                        "total_pago": "$pagamento.total_pago"
                    }
                }
            ]
        }
    }
]

res = list(db.pedidos.aggregate(pipeline_facet))[0]
print("KPIs:", res["kpis"])
print("Top Pedidos:", res["maiores_pedidos"])